## Carregamento de dataset de exemplo

Para exemplicação na disciplina, iremos trabalhar com o paradigma P300, utilizando o P300-Dataset do site [1].

Este dataset possui 8 participantes, e será carregado somente a base de dados de um deles para seguir com os exemplos.

A aplicação é de um speller (soletrador) 6x6, o qual foi utilizado nas seções experimentais, 7 palavras com 5 letras cada. Das 7 palavras, 3 foram utilizadas para treino e 4 para teste.

O equipamento utilizado foi o "g.Tec g.Nautilus g.LadyBird", com taxa de amostragem de 250 Hz, filtro `notch` de 50Hz (europeu) e filtro `bandpass` de 0.1 à 30 Hz.



In [1]:
# Carregamento do P300-dataset para o participante 8

from scipy.io import loadmat
import numpy as np

raw_data = loadmat('P300S08.mat')
raw_data.keys()

dict_keys(['__header__', '__version__', '__globals__', 'channelRange', 'Fs', 'Trials', 'stims', 'channelNames', 'sampleTime', 'samples', 'Word', 'data', 'subject'])

In [2]:
# Apresentando informações básicas presentes no dataset

# Taxa de amostragem
fs = raw_data['Fs'][0][0]
print(f'Taxa de amostragem: {fs} Hz')

# Canais utilizados
channels = raw_data['channelNames']
print('\nCanais utilizados: ', [str(name[0]) for name in channels[0]])
print('Número de canais: ', len(channels[0]))

Taxa de amostragem: 250 Hz

Canais utilizados:  ['FZ', 'CZ', 'P3', 'PZ', 'P4', 'PO7', 'PO8', 'OZ']
Número de canais:  8


In [3]:
# preparando os dados principais

main_data = raw_data['data'].copy()

X = main_data['X'][0][0]
y = main_data['y'][0][0].T[0]
stim = main_data['y_stim'][0][0].T[0]
trials = main_data['trial'][0][0][0]
flash = main_data['flash'][0][0]

# dados flutuantes / canais
print(X.shape)

# rótulos
print(y.shape)

# aparentemente, matriz com 0's contabilizando os estímulos
print(stim.shape)

# momento da amostragem onde cada um dos 35 ensaios começam
print(trials.shape)

# momento da amostragem onde cada flash é executado
# (ID da amostra, duração, estímulo, acerto/erro)
print(flash.shape)

(355904, 8)
(355904,)
(355904,)
(35,)
(4200, 4)


In [4]:
print(trials)
print(flash, len(flash))

print(y, len(y))
c0, c1, c2 = 0, 0, 0
for label in y:
    if label == 0:
        c0 += 1
    elif label == 1:
        c1 += 1
    else:
        c2 += 1
print(f"label 0: {c0}")
print(f"label 1: {c1}")
print(f"label 2: {c2}")

[  7478  17505  27533  37558  47580  57592  67628  77657  87682  97707
 107734 117759 127784 137808 147834 157860 167888 177910 187953 197955
 207987 218012 228037 238064 248088 258114 268138 278167 288194 298219
 308239 318275 328281 338317 348341]
[[  7478     32      7      1]
 [  7541     32      3      1]
 [  7604     32     10      1]
 ...
 [355682     31      3      2]
 [355745     31     10      1]
 [355807     31      4      1]] 4200
[0 0 0 ... 0 0 0] 355904
label 0: 235458
label 1: 100383
label 2: 20063


### Organização das informações da base de dados

Uma vez carregada e separada em variáveis distintas, é necessário agrupar os valores em apenas uma estrutura de dados (X) de forma consistente.

As próximas fases descrevem isto.

In [5]:
# dispensando dados que não ocorre estímulos (y == 0)

non_stim = np.where(y == 0)[0]
print(non_stim.shape)
print(X.shape)
print(y.shape)

y = np.delete(y, non_stim)
X = np.delete(X, non_stim, axis=0)
print(X.shape)
print(y.shape)

(235458,)
(355904, 8)
(355904,)
(120446, 8)
(120446,)


In [6]:
# separando os dados em blocos de flash
# atualização: como foram removidos todos os dados que não são estímulos, para os blocos de flash 
# serão considerados apenas os dados que possuem estímulos (y == 1 ou y == 2), dividos em tamanhos 
# +ou- iguais (tamanho = 32) às sequencias de y=2.
 
# verificando a quantidade média de y=2 entre os y=1, para definir o tamanho do bloco de flash
from statistics import mode

v2s = list()
v2 = 0
for v in y:
    if v == 1 and v2 != 0:
        v2s.append(v2)
        v2 = 0
    elif v == 2:
        v2 += 1

# definindo o tamanho do bloco de flash com base na média dos y=2
print(mode(v2s))

32


In [7]:
# criando o formato geral de "data_f" (blocos de flash) para os dados, com base na quantidade de y=2 entre os y=1

stim1 = np.where(y == 1)[0]
stim2 = np.where(y == 2)[0]

print("Soma de y=1 e y=2: ", len(stim1) + len(stim2))
print("Quantidade de y=1: ", len(stim1))
print("Quantidade de y=2: ", len(stim2))

# mostrando o resto da divisão da quantidade de y=2 e y=1 pelo tamanho do bloco (32)
print("Resto da divisão de y=2 por 32: ", len(stim2) % 32)
print("Resto da divisão de y=1 por 32: ", len(stim1) % 32)

# Dividindo os dados em amostras de 32, para cada sequência de y=2 entre os y=1, e armazenando em "data_f"
data_f1 = X[stim1]
data_f2 = X[stim2]

# adicionando uma linha de zeros no final de cada bloco, para que todos os blocos tenham o mesmo tamanho
data_f1 = np.insert(data_f1, data_f1.shape[0], 0, axis=0)
data_f2 = np.insert(data_f2, data_f2.shape[0], 0, axis=0)

# reorganizando os dados para o formato (número de blocos, tamanho do bloco, número de canais)
data_f1 = data_f1.reshape(-1, 32, data_f1.shape[1])
data_f2 = data_f2.reshape(-1, 32, data_f2.shape[1])

print(data_f1.shape)
print(data_f2.shape)

Soma de y=1 e y=2:  120446
Quantidade de y=1:  100383
Quantidade de y=2:  20063
Resto da divisão de y=2 por 32:  31
Resto da divisão de y=1 por 32:  31
(3137, 32, 8)
(627, 32, 8)


Por fim, vamos criar o array de dados final e um array de labels mais consistente para ter controle das classes onde o estímulo foi de acerto (1) ou não-acerto (0).

In [8]:
# concatenando os dados
data_f = np.concatenate((data_f1, data_f2), axis=0)

# criando o array de rótulos y correspondente
# usando 0 para a classe v1 e 1 para a classe v2
y_v1 = np.zeros(data_f1.shape[0], dtype=int)
y_v2 = np.ones(data_f2.shape[0], dtype=int)

# concatenando os rótulos y
data_y = np.concatenate((y_v1, y_v2), axis=0)

data_f.shape, data_y.shape

((3764, 32, 8), (3764,))

Concluimos então o formato dos dados como 3.764 amostras de 32 pontos flutuantes, para 8 eletrodos.

Temos dois conjuntos de classes, sendo 3.137 amostras com estímulos não corretos (fora da linha/coluna que o participante está vendo) e 627 com estímulos corretos (na linha/coluna que o participante está vendo).

In [9]:
# salvamento dos dados em numpy arrays

np.save('data_f_s08.npy', data_f)
np.save('data_y_s08.npy', data_y)

Agora iremos adaptar os dados a um objeto `MNE`. A ferramenta `MNE` permite organizar dados de EEG com mais informações de metadados, por exemplo, nome dos eletrodos e informações espaciais de sua distribuição.

Além disso, a ferramenta `MNE` disponibiliza métodos para facilitar **filtragem temporal** dos dados e impressão de gráficos de forma mais facilitade que o `matplotlib`.

In [10]:
# inicialmente criamos o objeto info
import mne

chs = [str(name[0]) for name in channels[0]]
n_channels = len(chs)
sfreq = int(fs)
ch_names = chs
ch_types = ['eeg'] * n_channels
info = mne.create_info(ch_names, sfreq=sfreq, ch_types=ch_types)
info

<Info | 7 non-empty values
 bads: []
 ch_names: FZ, CZ, P3, PZ, P4, PO7, PO8, OZ
 chs: 8 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 125.0 Hz
 meas_date: unspecified
 nchan: 8
 projs: []
 sfreq: 250.0 Hz
>

In [11]:
# agora vamos criar o objeto EpochsArray com os dados de EEG

print("Antes:", data_f.shape)
data_re = data_f.transpose(0, 2, 1)
print("Depois:", data_re.shape)

# criando o MNE events array
events = np.array([[i, 0, label] for i, label in enumerate(data_y)])
print(events.shape)

# criando o objeto propriamente dito
epochs = mne.EpochsArray(data_re, info, events=events, event_id={'miss': 0, 'hit': 1})

# salvando o objeto EpochsArray em um arquivo .fif
epochs.save('epochs_s08.fif', overwrite=True)

Antes: (3764, 32, 8)
Depois: (3764, 8, 32)
(3764, 3)
Not setting metadata
3764 matching events found
No baseline correction applied
0 projection items activated
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.


/tmp/ipykernel_12521/3236466490.py:15: RuntimeWarning: This filename (epochs_s08.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs.save('epochs_s08.fif', overwrite=True)


[PosixPath('/home/rhubner/tutoriais/SomeFeatures/epochs_s08.fif')]